# 03 — Entrenamiento del Modelo de Predicción de Tráfico

Entrenamiento, evaluación y comparativa de modelos para predecir el nivel de tráfico
(Bajo / Medio / Alto) en los distritos de Madrid según hora y tipo de día.

**Modelos evaluados:**
- Baseline: DummyClassifier
- Random Forest
- XGBoost

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

from ml.generate_dataset import generate_and_save

sns.set_theme(style='darkgrid')
RANDOM_STATE = 42

## 1. Carga y preparación del dataset

In [ ]:
DATASET_PATH = '../data/processed/aforos_dataset.csv'

if not os.path.exists(DATASET_PATH):
    generate_and_save()

df = pd.read_csv(DATASET_PATH)

le = LabelEncoder()
df['distrito_encoded'] = le.fit_transform(df['distrito'].astype(str))

FEATURES = ['hora', 'dia_semana', 'mes', 'es_fin_de_semana', 'es_festivo', 'distrito_encoded']
TARGET = 'nivel_trafico'

X = df[FEATURES].values
y = df[TARGET].values

print(f'Dataset: {X.shape[0]:,} muestras, {X.shape[1]} features')
print(f'Clases: {np.unique(y)} (0=Bajo, 1=Medio, 2=Alto)')
print(f'Distribución:\n{pd.Series(y).value_counts().sort_index()}')

## 2. Validación cruzada temporal

Se usa `TimeSeriesSplit` para respetar el orden temporal y evitar data leakage.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

modelos = {
    'Dummy (majority)': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
}

try:
    from xgboost import XGBClassifier
    modelos['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1
    )
except ImportError:
    print('XGBoost no instalado, se omite.')

resultados = {}
for nombre, clf in modelos.items():
    scores = cross_val_score(clf, X, y, cv=tscv, scoring='f1_macro', n_jobs=-1)
    resultados[nombre] = scores
    print(f'{nombre:25s}  F1-macro: {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Comparativa de modelos

In [ ]:
df_res = pd.DataFrame(resultados)

fig, ax = plt.subplots(figsize=(9, 4))
df_res.boxplot(ax=ax)
ax.set_title('F1-macro por fold (TimeSeriesSplit, 5 folds)')
ax.set_ylabel('F1-macro')
ax.set_ylim(0, 1)
ax.axhline(0.8, color='green', linestyle='--', alpha=0.7, label='Objetivo 80%')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Entrenamiento final y evaluación detallada

In [ ]:
# Entrenamiento en el 80% de los datos, evaluación en el 20% final (respetando orden temporal)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Bajo', 'Medio', 'Alto']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Bajo', 'Medio', 'Alto']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de confusión')

# Importancia de features
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Importancia de variables (Random Forest)')
axes[1].set_xlabel('Importancia media')

plt.tight_layout()
plt.show()

## 5. Guardar modelo final

In [ ]:
# Entrenar con TODOS los datos antes de guardar
rf_final = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
)
rf_final.fit(X, y)

os.makedirs('../data/processed', exist_ok=True)
joblib.dump(rf_final, '../data/processed/modelo_trafico_madrid.pkl')
joblib.dump(le, '../data/processed/encoder_madrid.pkl')
print('Modelo y encoder guardados en data/processed/')